> Versión de portafolio. Se conservan las salidas históricas del trabajo académico. Las rutas absolutas de datos, cuando existían, se sustituyeron por nombres locales. Esta versión no se ha ejecutado de nuevo; consulte README y DATOS.md antes de reproducir.


# Tarea M41 SQL con Python Javier Bueno Sanchez.

**Importación y consultas SQL con Python y SQLite.**

**Profesión Científico de Datos.**
**Actividad:** Creación de una base de datos SQLite y ejecución de consultas mediante SQL  
**Base de datos:** recursos_humanos.csv

**Preámbulo.**

En esta actividad se utilizarán Python y SQLite para crear y consultar una base de datos a partir del archivo externo `recursos_humanos.csv`, que contiene información sobre empleados de una empresa.

El desarrollo incluirá la importación y validación inicial de los datos, la creación del archivo `RH.db` y el almacenamiento de la información en una tabla denominada `Detalle`. Posteriormente, se realizarán consultas SQL con filtros y funciones de agregación para analizar la satisfacción laboral, las horas mensuales de trabajo, las promociones y las evaluaciones de los empleados.

El objetivo es integrar la capacidad de Python para administrar el flujo de trabajo con la eficiencia de SQL para extraer, filtrar y resumir información relevante.

**Índice.**

1. Importación de librerías.  
2. Carga del archivo CSV.  
3. Exploración y validación de los datos.  
4. Creación de la base de datos `RH.db`.  
5. Creación de la tabla `Detalle`.  
6. Validación de la importación.  
7. Consultas SQL solicitadas.  
   - 7.1 Satisfacción promedio según permanencia.  
   - 7.2 Horas promedio para salarios bajos y medios.  
   - 7.3 Empleados promovidos que abandonaron la empresa.  
   - 7.4 Empleados con evaluación igual o superior a 0.9.  
8. Cierre de la conexión.  
9. Conclusiones.

**1. Importación de librerías.** 

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


**2. Carga del archivo CSV.** 

In [2]:
ruta_csv = Path("recursos_humanos.csv")

if not ruta_csv.exists():
    raise FileNotFoundError(
        "No se encontró el archivo recursos_humanos.csv "
        "en la misma carpeta del notebook.")

df = pd.read_csv(ruta_csv)

print("Archivo cargado correctamente.")
print(f"Número de registros: {df.shape[0]:,}")
print(f"Número de columnas: {df.shape[1]}")

Archivo cargado correctamente.
Número de registros: 14,999
Número de columnas: 10


**3. Exploración y validación de los datos.**

Antes de crear la base de datos SQLite, se revisará la estructura del conjunto de datos para verificar sus dimensiones, nombres de columnas, tipos de datos, valores nulos y registros iniciales.

In [3]:
print("Dimensiones del conjunto de datos:")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")

print("\nNombres de las columnas:")
for columna in df.columns:
    print(f"- {columna}")

Dimensiones del conjunto de datos:
Filas: 14,999
Columnas: 10

Nombres de las columnas:
- satisfaction_level
- last_evaluation
- number_project
- average_montly_hours
- time_spend_company
- Work_accident
- left
- promotion_last_5years
- sales
- salary


In [4]:
display(df.head())

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [5]:
print("Tipos de datos:")
df.info()

print("\nValores nulos por columna:")
display(df.isnull().sum().to_frame(name="Valores nulos"))

Tipos de datos:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   sales                  14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(2), int64(6), object(2)
memory usage: 1.1+ MB

Valores nulos por columna:


,Valores nulos
satisfaction_level,0
last_evaluation,0
number_project,0
average_montly_hours,0
time_spend_company,0
Work_accident,0
left,0
promotion_last_5years,0
sales,0
salary,0


In [6]:
duplicados = df.duplicated().sum()

print(f"Número de registros duplicados: {duplicados:,}")

Número de registros duplicados: 3,008


**4. Creación de la base de datos `RH.db`.** 

Se establecerá una conexión con SQLite para crear el archivo `RH.db`. Posteriormente, el DataFrame se importará como una tabla denominada `Detalle`.

In [7]:
ruta_db = Path("RH.db")

conexion = sqlite3.connect(ruta_db)

df.to_sql(name="Detalle", con=conexion, if_exists="replace",index=False)

conexion.commit()

print(f"Base de datos creada correctamente: {ruta_db.resolve()}")
print("Tabla 'Detalle' creada e información importada correctamente.")

Base de datos creada correctamente: /home/javi/SQL/RH.db
Tabla 'Detalle' creada e información importada correctamente.


**5. Creación de la tabla `Detalle`.**

Se comprobará la existencia de la tabla `Detalle`, el número de registros importados y una muestra de su contenido.

In [8]:
consulta_validacion = """
SELECT COUNT(*) AS total_registros
FROM Detalle;
"""

total_registros = pd.read_sql_query(consulta_validacion, conexion)

display(total_registros)

,total_registros
0,14999


**6. Validación de la importación.** 

Se comprueba que los datos no solo existen en df, sino que fueron almacenados y recuperados correctamente desde la base de datos SQLite.

In [9]:
consulta_muestra = """
SELECT *
FROM Detalle
LIMIT 5;
"""

muestra_sql = pd.read_sql_query(consulta_muestra, conexion)

display(muestra_sql)

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


**7. Consultas SQL solicitadas.**

**7.1 Nivel de satisfacción promedio según permanencia.**

Se calculará el nivel de satisfacción promedio de los empleados agrupándolos según hayan permanecido en la empresa (`left = 0`) o la hayan abandonado (`left = 1`). El grupo con el promedio más alto será el resultado de interés.

In [10]:
consulta_satisfaccion = """
SELECT
    "left" AS abandono_empresa,
    COUNT(*) AS numero_empleados,
    ROUND(AVG(satisfaction_level), 4) AS satisfaccion_promedio
FROM Detalle
GROUP BY "left"
ORDER BY satisfaccion_promedio DESC;
"""

resultado_satisfaccion = pd.read_sql_query(
    consulta_satisfaccion,
    conexion
)

display(resultado_satisfaccion)

,abandono_empresa,numero_empleados,satisfaccion_promedio
0,0,11428,0.6668
1,1,3571,0.4401


**Interpretación.**

Los empleados que permanecieron en la empresa (`left = 0`) presentan un nivel de satisfacción promedio de **0.6668**, mientras que quienes abandonaron la empresa (`left = 1`) registran un promedio de **0.4401**.

Por tanto, el nivel de satisfacción promedio más alto corresponde a los empleados que decidieron permanecer en la empresa. La diferencia es de **0.2267 puntos**, lo que evidencia una asociación clara entre una menor satisfacción laboral y el abandono de la organización.

**7.2 Promedio de horas mensuales según el nivel salarial.**

Se calculará el promedio de horas mensuales trabajadas por los empleados con salario bajo (`low`) o medio (`medium`). Los resultados se presentarán por nivel salarial para facilitar su comparación.

In [11]:
consulta_horas_salario = """
SELECT
    salary AS nivel_salario,
    COUNT(*) AS numero_empleados,
    ROUND(AVG(average_montly_hours), 2) AS horas_mensuales_promedio
FROM Detalle
WHERE salary IN ('low', 'medium')
GROUP BY salary
ORDER BY horas_mensuales_promedio DESC;
"""

resultado_horas_salario = pd.read_sql_query(consulta_horas_salario, conexion)

display(resultado_horas_salario)

,nivel_salario,numero_empleados,horas_mensuales_promedio
0,medium,6446,201.34
1,low,7316,201.00


**Interpretación por nivel salarial.**

Los empleados con salario medio trabajan un promedio de **201.34 horas mensuales**, mientras que aquellos con salario bajo trabajan **201.00 horas mensuales**.

La diferencia es de apenas **0.34 horas mensuales**, por lo que el nivel salarial bajo o medio no presenta una diferencia relevante en la carga horaria promedio.

In [12]:
consulta_horas_general = """
SELECT
    COUNT(*) AS numero_empleados,
    ROUND(AVG(average_montly_hours), 2) AS horas_mensuales_promedio
FROM Detalle
WHERE salary IN ('low', 'medium');
"""

resultado_horas_general = pd.read_sql_query(consulta_horas_general, conexion)

display(resultado_horas_general)

,numero_empleados,horas_mensuales_promedio
0,13762,201.16


**Conclusión.**

Considerando conjuntamente a los empleados con nivel salarial bajo (`low`) y medio (`medium`), se analizaron **13,762 empleados**, quienes trabajan un promedio de **201.16 horas mensuales**.

Al segmentar los resultados, los empleados con salario medio registran **201.34 horas**, mientras que aquellos con salario bajo registran **201.00 horas mensuales**. La diferencia de **0.34 horas** es mínima, por lo que ambos grupos presentan una carga horaria promedio prácticamente equivalente.

**7.3 Empleados promovidos que abandonaron la empresa.**

Se extraerán los registros de los empleados que recibieron una promoción durante los últimos cinco años y que, además, abandonaron la empresa.

In [13]:
consulta_promocion_abandono = """
SELECT *
FROM Detalle
WHERE promotion_last_5years = 1
  AND "left" = 1;
"""

resultado_promocion_abandono = pd.read_sql_query(consulta_promocion_abandono, conexion)

print("Número de empleados promovidos que abandonaron la empresa:",
    len(resultado_promocion_abandono))

display(resultado_promocion_abandono)

Número de empleados promovidos que abandonaron la empresa: 19


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.45,0.51,2,160,3,1,1,1,sales,low
1,0.79,0.59,4,139,3,0,1,1,management,low
2,0.41,0.46,2,160,3,0,1,1,sales,low
3,0.11,0.79,6,292,4,0,1,1,technical,low
4,0.41,0.56,2,154,3,0,1,1,support,medium
5,0.46,0.45,2,138,3,0,1,1,IT,low
6,0.87,1.00,4,258,5,1,1,1,sales,medium
7,0.44,0.55,2,128,3,0,1,1,IT,medium
8,0.45,0.51,2,160,3,1,1,1,sales,low
9,0.79,0.59,4,139,3,0,1,1,management,low


**Interpretación.**

La consulta identificó **19 empleados** que recibieron una promoción durante los últimos cinco años y que, aun así, abandonaron la empresa.

Este resultado demuestra que una promoción no garantiza por sí sola la permanencia del empleado. Para explicar estas salidas sería necesario analizar conjuntamente otras variables, como el nivel de satisfacción, la carga de trabajo, el salario, la antigüedad y la evaluación de desempeño.

**7.4 Empleados con una evaluación igual o superior a 0.9.**

Se extraerán los registros correspondientes a los empleados que obtuvieron una calificación de **0.9 o más** en su última evaluación.

In [14]:
consulta_evaluacion_alta = """
SELECT *
FROM Detalle
WHERE last_evaluation >= 0.9
ORDER BY last_evaluation DESC;
"""

resultado_evaluacion_alta = pd.read_sql_query(consulta_evaluacion_alta, conexion)

print("Número de empleados con evaluación igual o superior a 0.9:",
    len(resultado_evaluacion_alta))

display(resultado_evaluacion_alta)

Número de empleados con evaluación igual o superior a 0.9: 2988


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,sales,salary
0,0.89,1.0,5,224,5,0,1,0,sales,low
1,0.85,1.0,4,225,5,0,1,0,technical,low
2,0.89,1.0,5,246,5,0,1,0,sales,low
3,0.90,1.0,5,221,6,0,1,0,sales,medium
4,0.91,1.0,4,257,5,0,1,0,accounting,medium
...,...,...,...,...,...,...,...,...,...,...
2983,0.10,0.9,7,286,4,0,1,0,sales,low
2984,0.10,0.9,6,299,4,0,1,0,marketing,low
2985,0.11,0.9,6,253,4,0,1,0,sales,medium
2986,0.10,0.9,7,269,4,0,1,0,accounting,medium


**Interpretación.**

La consulta identificó **2,988 empleados** cuya última evaluación fue igual o superior a **0.9**. Estos registros representan al grupo con las calificaciones de desempeño más altas dentro del conjunto de datos.

El resultado permite localizar empleados con un desempeño destacado, aunque una evaluación elevada no debe interpretarse de forma aislada como garantía de satisfacción o permanencia en la empresa.

**8. Cierre de la conexión.**

Una vez finalizadas y validadas las consultas, se cerrará la conexión con la base de datos para liberar correctamente los recursos utilizados.

In [15]:
conexion.close()

print("Conexión con la base de datos RH.db cerrada correctamente.")

Conexión con la base de datos RH.db cerrada correctamente.


**9. Conclusiones.**

En esta actividad se creó correctamente la base de datos SQLite `RH.db` a partir del archivo externo `recursos_humanos.csv`. La información de **14,999 empleados** fue almacenada en la tabla `Detalle` y posteriormente consultada mediante instrucciones SQL ejecutadas desde Python.

Los principales resultados fueron los siguientes:

- Los empleados que permanecieron en la empresa presentaron una satisfacción promedio de **0.6668**, superior al **0.4401** registrado por quienes abandonaron la organización.
- Los **13,762 empleados** con salario bajo o medio trabajaron un promedio conjunto de **201.16 horas mensuales**.
- Por nivel salarial, el promedio fue de **201.00 horas** para salarios bajos y **201.34 horas** para salarios medios, mostrando una diferencia mínima.
- Se identificaron **19 empleados** que recibieron una promoción durante los últimos cinco años y que, aun así, abandonaron la empresa.
- Se localizaron **2,988 empleados** cuya última evaluación fue igual o superior a **0.9**.

Los resultados muestran que una menor satisfacción laboral está asociada con el abandono de la empresa. También evidencian que variables como una promoción o una evaluación elevada no explican por sí solas la permanencia del empleado, por lo que deben analizarse junto con el salario, la carga de trabajo y la antigüedad.

La actividad permitió integrar Python como herramienta de administración y presentación de resultados con SQLite como motor para almacenar, filtrar y resumir los datos mediante SQL.